# Advanced Python Set Problems — A Step-by-Step Tutorial

This notebook continues the study of Python sets through a new collection of advanced problems.

The presentation deliberately follows a tutorial style:

- ideas are introduced in small logical steps,
- each problem starts with a concrete scenario,
- intermediate results are inspected,
- complete solutions are built gradually,
- and assertions are used to verify correctness.

The main goal is not only to obtain the final answer, but also to understand why a set is the right tool for each problem.

## What We Will Practice

We will work with:

- membership testing,
- duplicate detection,
- set comprehensions,
- unions and intersections,
- differences and symmetric differences,
- subset and superset checks,
- disjointness,
- safe mutation,
- `frozenset`,
- graph algorithms,
- permission systems,
- text similarity,
- custom hashable objects,
- performance and memory trade-offs.

## A Reminder About Set Order

A set is an unordered collection of unique hashable objects.

You should never write code that depends on the displayed or iteration order of a set.

Whenever an example needs stable output, we will display values using `sorted(...)`.

## Imports

In [1]:
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from itertools import combinations
from statistics import median
from timeit import repeat
import re
import sys

# Problem 1 — Analyze Duplicate Events

Suppose an application receives a stream of event IDs.

An event may arrive more than once because of retries, network duplication, or a producer bug.

We want to determine:

- which events are unique,
- which events are duplicates,
- and which event is duplicated first.

## Step 1: Start With a Small Event Stream

The set `seen` will contain every event ID encountered so far.

The set `duplicates` will contain IDs that appear more than once.

In [2]:
events = [
    "evt-101",
    "evt-102",
    "evt-103",
    "evt-102",
    "evt-104",
    "evt-101",
    "evt-105",
]

seen = set()
duplicates = set()

for event_id in events:
    if event_id in seen:
        duplicates.add(event_id)
    else:
        seen.add(event_id)

print("Seen:", sorted(seen))
print("Duplicates:", sorted(duplicates))

Seen: ['evt-101', 'evt-102', 'evt-103', 'evt-104', 'evt-105']
Duplicates: ['evt-101', 'evt-102']


The set of seen values contains every distinct event ID.

The duplicate set contains only IDs for which at least one repeated occurrence was found.

## Step 2: Record the First Repeated Event

We now need slightly more state.

When the first duplicate is found, we store both the event ID and its position.

In [3]:
seen = set()
first_repeat = None

for index, event_id in enumerate(events):
    if event_id in seen:
        first_repeat = (event_id, index)
        break
    seen.add(event_id)

print(first_repeat)

('evt-102', 3)


The index is the position of the repeated occurrence, not the first occurrence.

In this example, `evt-102` first appears at index `1` and repeats at index `3`.

## Step 3: Build a Reusable Solution

The function below returns a complete report.

It also returns the number of distinct values.

In [4]:
def analyze_duplicates(items):
    seen = set()
    duplicates = set()
    first_repeat = None

    for index, item in enumerate(items):
        if item in seen:
            duplicates.add(item)
            if first_repeat is None:
                first_repeat = (item, index)
        else:
            seen.add(item)

    return {
        "distinct": seen,
        "duplicates": duplicates,
        "first_repeat": first_repeat,
        "distinct_count": len(seen),
    }


duplicate_report = analyze_duplicates(events)

assert duplicate_report["distinct"] == {
    "evt-101", "evt-102", "evt-103", "evt-104", "evt-105"
}
assert duplicate_report["duplicates"] == {"evt-101", "evt-102"}
assert duplicate_report["first_repeat"] == ("evt-102", 3)
assert duplicate_report["distinct_count"] == 5

duplicate_report

{'distinct': {'evt-101', 'evt-102', 'evt-103', 'evt-104', 'evt-105'},
 'duplicates': {'evt-101', 'evt-102'},
 'first_repeat': ('evt-102', 3),
 'distinct_count': 5}

## Complexity

Each event is checked once.

Average-case set membership is constant time, so the complete scan is expected to take:

- time: `O(n)`
- additional space: `O(k)`

Here, `k` is the number of distinct event IDs.

# Problem 2 — Stable Deduplication With Normalization

A customer database contains email addresses entered with inconsistent capitalization and accidental spaces.

We want to:

1. normalize every address,
2. remove duplicates,
3. preserve the first original spelling encountered.

## Step 1: Define a Normalization Rule

For this exercise, normalization means:

- remove leading and trailing whitespace,
- convert letters to lowercase.

In [5]:
def normalize_email(email):
    return email.strip().lower()


assert normalize_email("  Ada@Example.COM ") == "ada@example.com"

## Step 2: Understand Why `set(emails)` Is Not Enough

A set can remove exact duplicates, but it does not automatically apply normalization.

It also does not preserve the first original representation as an ordered result.

In [6]:
raw_emails = [
    "Ada@Example.com",
    " grace@example.com ",
    "ADA@example.com",
    "linus@example.com",
    "Grace@Example.Com",
]

exact_unique = set(raw_emails)
print("Exact unique count:", len(exact_unique))
print("Normalized unique count:", len({normalize_email(email) for email in raw_emails}))

Exact unique count: 5
Normalized unique count: 3


## Step 3: Use a Set for Lookup and a List for Output

This is a common pattern:

- the set answers, “Have we already seen this normalized value?”
- the list preserves the desired result order.

In [7]:
def unique_emails_in_order(emails):
    seen_normalized = set()
    result = []

    for original_email in emails:
        normalized = normalize_email(original_email)

        if normalized not in seen_normalized:
            seen_normalized.add(normalized)
            result.append(original_email.strip())

    return result


unique_emails = unique_emails_in_order(raw_emails)

assert unique_emails == [
    "Ada@Example.com",
    "grace@example.com",
    "linus@example.com",
]

unique_emails

['Ada@Example.com', 'grace@example.com', 'linus@example.com']

This solution demonstrates that a set does not need to be the final output.

Sets are often most useful as supporting data structures.

# Problem 3 — Compare Multiple User Groups

A learning platform has several course groups.

We want to find:

- users in every group,
- users in at least one group,
- users in exactly one group,
- users in more than one group.

## Step 1: Create the Groups

In [8]:
python_group = {"Ada", "Grace", "Linus", "Guido"}
database_group = {"Grace", "Linus", "Margaret", "Edsger"}
algorithms_group = {"Ada", "Grace", "Edsger", "Donald"}

print("Python:", sorted(python_group))
print("Databases:", sorted(database_group))
print("Algorithms:", sorted(algorithms_group))

Python: ['Ada', 'Grace', 'Guido', 'Linus']
Databases: ['Edsger', 'Grace', 'Linus', 'Margaret']
Algorithms: ['Ada', 'Donald', 'Edsger', 'Grace']


## Step 2: Find Users in Every Group

The intersection contains values shared by all sets.

In [9]:
in_every_group = python_group & database_group & algorithms_group

assert in_every_group == {"Grace"}
sorted(in_every_group)

['Grace']

We can also call `set.intersection` with several sets.

This is useful when the number of groups is dynamic.

In [10]:
groups = [python_group, database_group, algorithms_group]

same_result = set.intersection(*groups)

assert same_result == in_every_group

## Step 3: Find Users in At Least One Group

The union contains every distinct user present in any group.

In [11]:
in_any_group = set.union(*groups)

assert in_any_group == {
    "Ada", "Grace", "Linus", "Guido",
    "Margaret", "Edsger", "Donald"
}

sorted(in_any_group)

['Ada', 'Donald', 'Edsger', 'Grace', 'Guido', 'Linus', 'Margaret']

## Step 4: Count Group Memberships

Set algebra alone can identify many relationships.

For “exactly one group,” counting memberships is especially clear.

In [12]:
membership_counts = Counter(
    user
    for group in groups
    for user in group
)

exactly_one_group = {
    user
    for user, count in membership_counts.items()
    if count == 1
}

more_than_one_group = {
    user
    for user, count in membership_counts.items()
    if count > 1
}

print("Exactly one:", sorted(exactly_one_group))
print("More than one:", sorted(more_than_one_group))

Exactly one: ['Donald', 'Guido', 'Margaret']
More than one: ['Ada', 'Edsger', 'Grace', 'Linus']


## Step 5: Package the Logic

The function below accepts any mapping of group names to iterables.

In [13]:
def analyze_groups(named_groups):
    group_sets = {
        name: set(members)
        for name, members in named_groups.items()
    }

    if not group_sets:
        return {
            "everywhere": set(),
            "anywhere": set(),
            "exactly_one": set(),
            "multiple": set(),
        }

    sets = list(group_sets.values())
    everywhere = set.intersection(*sets)
    anywhere = set.union(*sets)

    counts = Counter(
        member
        for group in sets
        for member in group
    )

    exactly_one = {member for member, count in counts.items() if count == 1}
    multiple = {member for member, count in counts.items() if count > 1}

    return {
        "everywhere": everywhere,
        "anywhere": anywhere,
        "exactly_one": exactly_one,
        "multiple": multiple,
    }


group_report = analyze_groups({
    "python": python_group,
    "databases": database_group,
    "algorithms": algorithms_group,
})

assert group_report["everywhere"] == {"Grace"}
assert group_report["exactly_one"] == {"Guido", "Margaret", "Donald"}
assert group_report["multiple"] == {"Ada", "Grace", "Linus", "Edsger"}

group_report

{'everywhere': {'Grace'},
 'anywhere': {'Ada',
  'Donald',
  'Edsger',
  'Grace',
  'Guido',
  'Linus',
  'Margaret'},
 'exactly_one': {'Donald', 'Guido', 'Margaret'},
 'multiple': {'Ada', 'Edsger', 'Grace', 'Linus'}}

# Problem 4 — Validate Role-Based Permissions

A user receives permissions from one or more roles.

The system also supports:

- explicit permission grants,
- explicit permission denials,
- required permissions,
- and detection of unknown roles.

## Step 1: Define Roles

Each role maps to a set of permissions.

In [14]:
role_permissions = {
    "viewer": {"read"},
    "editor": {"read", "write", "comment"},
    "auditor": {"read", "export"},
    "administrator": {"read", "write", "comment", "export", "delete"},
}

## Step 2: Combine Role Permissions

Permissions from multiple roles are combined using a union.

In [15]:
assigned_roles = {"editor", "auditor"}

combined_role_permissions = set().union(
    *(role_permissions[role] for role in assigned_roles)
)

assert combined_role_permissions == {"read", "write", "comment", "export"}
sorted(combined_role_permissions)

['comment', 'export', 'read', 'write']

## Step 3: Apply Explicit Grants and Denials

A grant adds a permission.

A denial removes a permission even if a role grants it.

In [16]:
explicit_grants = {"download"}
explicit_denials = {"write"}

effective_permissions = (
    combined_role_permissions
    | explicit_grants
) - explicit_denials

assert effective_permissions == {"read", "comment", "export", "download"}
sorted(effective_permissions)

['comment', 'download', 'export', 'read']

## Step 4: Check Required Permissions

The expression `required - effective` finds every required permission that is missing.

In [17]:
required_permissions = {"read", "download"}

missing_required = required_permissions - effective_permissions

assert missing_required == set()

## Step 5: Write the Complete Evaluator

The final permission snapshot is returned as a `frozenset`.

This communicates that callers should treat the result as immutable.

In [18]:
def evaluate_permissions(
    assigned_roles,
    role_permissions,
    *,
    explicit_grants=frozenset(),
    explicit_denials=frozenset(),
    required=frozenset(),
):
    assigned_roles = set(assigned_roles)
    known_roles = set(role_permissions)

    unknown_roles = assigned_roles - known_roles
    if unknown_roles:
        raise ValueError(f"Unknown roles: {sorted(unknown_roles)}")

    role_grants = set().union(
        *(role_permissions[role] for role in assigned_roles)
    ) if assigned_roles else set()

    explicit_grants = set(explicit_grants)
    explicit_denials = set(explicit_denials)
    required = set(required)

    effective = (role_grants | explicit_grants) - explicit_denials
    missing = required - effective
    redundant_grants = explicit_grants & role_grants
    denied_role_permissions = explicit_denials & role_grants

    return {
        "effective": frozenset(effective),
        "missing_required": missing,
        "redundant_grants": redundant_grants,
        "denied_role_permissions": denied_role_permissions,
        "valid": not missing,
    }


permission_report = evaluate_permissions(
    {"editor", "auditor"},
    role_permissions,
    explicit_grants={"download", "export"},
    explicit_denials={"write"},
    required={"read", "download"},
)

assert permission_report["effective"] == frozenset(
    {"read", "comment", "export", "download"}
)
assert permission_report["redundant_grants"] == {"export"}
assert permission_report["denied_role_permissions"] == {"write"}
assert permission_report["valid"] is True

permission_report

{'effective': frozenset({'comment', 'download', 'export', 'read'}),
 'missing_required': set(),
 'redundant_grants': {'export'},
 'denied_role_permissions': {'write'},
 'valid': True}

# Problem 5 — Measure Text Similarity With Jaccard Similarity

A simple way to compare two documents is to represent each document as a set of normalized words.

The Jaccard similarity is:

\[
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
\]

A score of:

- `1.0` means the sets are identical,
- `0.0` means they have no shared elements.

## Step 1: Normalize Text Into a Set of Words

In [19]:
def words(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))


text_a = "Python sets provide fast membership testing."
text_b = "Membership testing with Python sets is fast."

words_a = words(text_a)
words_b = words(text_b)

print(sorted(words_a))
print(sorted(words_b))

['fast', 'membership', 'provide', 'python', 'sets', 'testing']
['fast', 'is', 'membership', 'python', 'sets', 'testing', 'with']


## Step 2: Inspect the Intersection and Union

In [20]:
shared_words = words_a & words_b
all_words = words_a | words_b

print("Shared:", sorted(shared_words))
print("All:", sorted(all_words))
print("Shared count:", len(shared_words))
print("Union count:", len(all_words))

Shared: ['fast', 'membership', 'python', 'sets', 'testing']
All: ['fast', 'is', 'membership', 'provide', 'python', 'sets', 'testing', 'with']
Shared count: 5
Union count: 8


## Step 3: Handle the Empty-Set Edge Case

If both inputs are empty after normalization, the mathematical formula produces `0 / 0`.

For this application, we will define two empty documents as perfectly similar.

In [21]:
def jaccard_similarity(left, right):
    left = set(left)
    right = set(right)

    union = left | right

    if not union:
        return 1.0

    return len(left & right) / len(union)


score = jaccard_similarity(words_a, words_b)

assert 0.0 <= score <= 1.0
assert jaccard_similarity(set(), set()) == 1.0
assert jaccard_similarity({"a"}, {"b"}) == 0.0
assert jaccard_similarity({"a", "b"}, {"a", "b"}) == 1.0

score

0.625

## Step 4: Compare Several Documents

We can now rank documents by similarity to a query.

In [22]:
documents = {
    "doc-1": "Python sets provide fast membership tests.",
    "doc-2": "Lists preserve order and allow duplicates.",
    "doc-3": "Set membership in Python is usually fast.",
    "doc-4": "Database indexes support efficient lookup.",
}

query = "fast Python set membership"
query_words = words(query)

ranking = sorted(
    (
        (
            document_id,
            jaccard_similarity(query_words, words(text)),
        )
        for document_id, text in documents.items()
    ),
    key=lambda item: item[1],
    reverse=True,
)

ranking

[('doc-3', 0.5714285714285714),
 ('doc-1', 0.42857142857142855),
 ('doc-2', 0.0),
 ('doc-4', 0.0)]

# Problem 6 — Build a Search Index

An inverted index maps each word to the set of documents containing that word.

This structure makes multi-keyword search a natural set-algebra problem.

## Step 1: Build the Index

Each word becomes a dictionary key.

The associated value is a set of document IDs.

In [23]:
def build_inverted_index(documents):
    index = defaultdict(set)

    for document_id, text in documents.items():
        for word in words(text):
            index[word].add(document_id)

    return dict(index)


index = build_inverted_index(documents)

print("python:", sorted(index.get("python", set())))
print("membership:", sorted(index.get("membership", set())))

python: ['doc-1', 'doc-3']
membership: ['doc-1', 'doc-3']


## Step 2: Find Documents Matching Every Search Term

For every query term, retrieve its posting set.

Then intersect all posting sets.

In [24]:
query_terms = {"python", "membership"}

postings = [
    index.get(term, set())
    for term in query_terms
]

match_all = set.intersection(*postings)

assert match_all == {"doc-1", "doc-3"}
sorted(match_all)

['doc-1', 'doc-3']

## Step 3: Find Documents Matching Any Search Term

For an OR-style search, use the union of posting sets.

In [25]:
match_any = set.union(*postings)

assert match_any == {"doc-1", "doc-3"}
sorted(match_any)

['doc-1', 'doc-3']

## Step 4: Handle an Empty Query

We must define what an empty query means.

In this tutorial:

- matching all zero terms returns all documents,
- matching any zero terms returns no documents.

In [26]:
def search_index(index, all_document_ids, query_terms):
    query_terms = {term.lower() for term in query_terms}
    all_document_ids = set(all_document_ids)

    postings = [
        index.get(term, set())
        for term in query_terms
    ]

    match_all = (
        set.intersection(*postings)
        if postings
        else all_document_ids.copy()
    )

    match_any = (
        set.union(*postings)
        if postings
        else set()
    )

    match_none = all_document_ids - match_any

    return {
        "all": match_all,
        "any": match_any,
        "none": match_none,
    }


search_report = search_index(
    index,
    documents,
    {"python", "membership"},
)

assert search_report["all"] == {"doc-1", "doc-3"}
assert search_report["none"] == {"doc-2", "doc-4"}

search_report

{'all': {'doc-1', 'doc-3'},
 'any': {'doc-1', 'doc-3'},
 'none': {'doc-2', 'doc-4'}}

# Problem 7 — Reconcile Two Inventories

Two systems contain product IDs:

- the warehouse system,
- the online store.

We need to identify:

- products present in both systems,
- products missing from the online store,
- products unexpectedly present online,
- every product involved in a mismatch.

## Step 1: Create the Two Snapshots

In [27]:
warehouse_products = {
    "P-100", "P-101", "P-102", "P-103", "P-104"
}

online_products = {
    "P-100", "P-102", "P-104", "P-105"
}

## Step 2: Products Present in Both Systems

This is the intersection.

In [28]:
present_in_both = warehouse_products & online_products

assert present_in_both == {"P-100", "P-102", "P-104"}
sorted(present_in_both)

['P-100', 'P-102', 'P-104']

## Step 3: Directional Differences

A difference is directional.

`warehouse - online` is not the same as `online - warehouse`.

In [29]:
missing_online = warehouse_products - online_products
unexpected_online = online_products - warehouse_products

assert missing_online == {"P-101", "P-103"}
assert unexpected_online == {"P-105"}

print("Missing online:", sorted(missing_online))
print("Unexpected online:", sorted(unexpected_online))

Missing online: ['P-101', 'P-103']
Unexpected online: ['P-105']


## Step 4: All Mismatches

The symmetric difference contains elements in exactly one of the two sets.

In [30]:
all_mismatches = warehouse_products ^ online_products

assert all_mismatches == {"P-101", "P-103", "P-105"}
sorted(all_mismatches)

['P-101', 'P-103', 'P-105']

## Step 5: Create a Reusable Reconciliation Function

In [31]:
def reconcile_sets(expected, observed):
    expected = set(expected)
    observed = set(observed)

    missing = expected - observed
    unexpected = observed - expected
    mismatched = expected ^ observed

    return {
        "matches": expected & observed,
        "missing": missing,
        "unexpected": unexpected,
        "mismatched": mismatched,
        "in_sync": not mismatched,
    }


inventory_report = reconcile_sets(
    warehouse_products,
    online_products,
)

assert inventory_report["in_sync"] is False
assert inventory_report["mismatched"] == {
    "P-101", "P-103", "P-105"
}

inventory_report

{'matches': {'P-100', 'P-102', 'P-104'},
 'missing': {'P-101', 'P-103'},
 'unexpected': {'P-105'},
 'mismatched': {'P-101', 'P-103', 'P-105'},
 'in_sync': False}

# Problem 8 — Remove Values Safely

Set mutation is simple, but the choice of method matters.

We will compare:

- `remove`,
- `discard`,
- `pop`,
- `difference_update`.

## Step 1: `remove` Enforces a Strict Contract

`remove(value)` raises `KeyError` when the value is absent.

Use it when absence indicates an invalid state or programming error.

In [32]:
active_users = {"Ada", "Grace", "Linus"}

active_users.remove("Grace")

assert active_users == {"Ada", "Linus"}

try:
    active_users.remove("Missing")
except KeyError:
    print("Strict removal correctly raised KeyError.")

Strict removal correctly raised KeyError.


## Step 2: `discard` Is Forgiving

`discard(value)` silently does nothing when the value is absent.

Use it when repeated deletion is acceptable.

In [33]:
active_users.discard("Missing")
active_users.discard("Linus")

assert active_users == {"Ada"}

## Step 3: Remove Several Values Efficiently

`difference_update` mutates the set in place.

In [34]:
active_sessions = {"s1", "s2", "s3", "s4", "s5"}
revoked_sessions = {"s2", "s4", "s9"}

active_sessions.difference_update(revoked_sessions)

assert active_sessions == {"s1", "s3", "s5"}
active_sessions

{'s1', 's3', 's5'}

## Step 4: Do Not Change a Set's Size While Iterating Over It

The following pattern is unsafe:

```python
for value in values:
    values.remove(value)
```

Instead, either create a new set or iterate over a copy.

In [35]:
values = {1, 2, 3, 4, 5, 6}

# Safe approach: construct a filtered set.
odd_values = {
    value
    for value in values
    if value % 2 == 1
}

assert odd_values == {1, 3, 5}

In [36]:
values = {1, 2, 3, 4, 5, 6}

# Also safe: iterate over a snapshot.
for value in values.copy():
    if value % 2 == 0:
        values.remove(value)

assert values == {1, 3, 5}

## Step 5: Understand `pop`

`set.pop()` removes an arbitrary element.

It should only be used when order does not matter.

In [37]:
pending = {"task-A", "task-B", "task-C"}
processed = []

while pending:
    processed.append(pending.pop())

assert set(processed) == {"task-A", "task-B", "task-C"}
assert pending == set()

print("Arbitrary processing order:", processed)

Arbitrary processing order: ['task-B', 'task-A', 'task-C']


# Problem 9 — Use `frozenset` for Unordered Composite Keys

A normal set is mutable and therefore unhashable.

This means a set cannot be:

- an element of another set,
- or a dictionary key.

A `frozenset` is immutable and hashable.

## Step 1: Represent an Undirected Connection

The connection between `A` and `B` is the same as the connection between `B` and `A`.

A `frozenset` naturally captures that idea.

In [38]:
edge_1 = frozenset({"A", "B"})
edge_2 = frozenset({"B", "A"})

assert edge_1 == edge_2
assert hash(edge_1) == hash(edge_2)

## Step 2: Deduplicate Undirected Edges

Self-loops will be ignored.

In [39]:
def unique_undirected_edges(edges):
    result = set()

    for left, right in edges:
        if left == right:
            continue

        result.add(frozenset({left, right}))

    return result


raw_edges = [
    ("A", "B"),
    ("B", "A"),
    ("A", "C"),
    ("C", "A"),
    ("B", "C"),
    ("B", "B"),
]

unique_edges = unique_undirected_edges(raw_edges)

assert len(unique_edges) == 3
unique_edges

{frozenset({'B', 'C'}), frozenset({'A', 'C'}), frozenset({'A', 'B'})}

## Step 3: Use a `frozenset` as a Dictionary Key

This is useful when the identity of a group does not depend on order.

In [40]:
route_costs = {
    frozenset({"Sofia", "Plovdiv"}): 15,
    frozenset({"Sofia", "Varna"}): 30,
}

assert route_costs[frozenset({"Plovdiv", "Sofia"})] == 15

# Problem 10 — Find Common Neighbors in a Graph

A graph can be represented as a dictionary mapping each node to a set of neighboring nodes.

Set intersections then reveal shared neighbors.

## Step 1: Build the Adjacency Sets

In [41]:
graph = {
    "A": {"B", "C", "D"},
    "B": {"A", "C", "D"},
    "C": {"A", "B", "D", "E"},
    "D": {"A", "B", "C"},
    "E": {"C"},
}

## Step 2: Find Common Neighbors of Two Nodes

In [42]:
common_ab = graph["A"] & graph["B"]

assert common_ab == {"C", "D"}
sorted(common_ab)

['C', 'D']

## Step 3: Find Triangles

A triangle consists of three nodes that are pairwise connected.

For each edge `(u, v)`, every common neighbor of `u` and `v` completes a triangle.

In [43]:
def graph_edges(adjacency):
    edges = set()

    for node, neighbors in adjacency.items():
        for neighbor in neighbors:
            edges.add(frozenset({node, neighbor}))

    return {
        edge
        for edge in edges
        if len(edge) == 2
    }


edges = graph_edges(graph)
len(edges)

7

In [44]:
def find_triangles(adjacency):
    triangles = set()

    for edge in graph_edges(adjacency):
        left, right = tuple(edge)

        for common_neighbor in adjacency[left] & adjacency[right]:
            triangle = frozenset({
                left,
                right,
                common_neighbor,
            })

            if len(triangle) == 3:
                triangles.add(triangle)

    return triangles


triangles = find_triangles(graph)

expected_triangles = {
    frozenset({"A", "B", "C"}),
    frozenset({"A", "B", "D"}),
    frozenset({"A", "C", "D"}),
    frozenset({"B", "C", "D"}),
}

assert triangles == expected_triangles
triangles

{frozenset({'A', 'B', 'C'}),
 frozenset({'A', 'C', 'D'}),
 frozenset({'B', 'C', 'D'}),
 frozenset({'A', 'B', 'D'})}

Because each triangle is stored as a `frozenset`, the same triangle found through different edges is automatically deduplicated.

# Problem 11 — Breadth-First Search With a Visited Set

Graph traversal requires a way to avoid visiting the same node repeatedly.

A set is ideal for the visited collection.

## Step 1: Understand the Queue and the Set

The queue controls traversal order.

The set answers whether a node has already been discovered.

In [45]:
def breadth_first_order(adjacency, start):
    if start not in adjacency:
        return []

    queue = deque([start])
    visited = {start}
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)

        for neighbor in sorted(adjacency[node]):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return order


bfs_order = breadth_first_order(graph, "A")

assert bfs_order[0] == "A"
assert set(bfs_order) == set(graph)
bfs_order

['A', 'B', 'C', 'D', 'E']

## Step 2: Find All Reachable Nodes

If only reachability matters, we can return the visited set.

In [46]:
def reachable_nodes(adjacency, start):
    if start not in adjacency:
        return set()

    queue = deque([start])
    visited = {start}

    while queue:
        node = queue.popleft()

        for neighbor in adjacency[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return visited


assert reachable_nodes(graph, "E") == {"A", "B", "C", "D", "E"}
assert reachable_nodes(graph, "missing") == set()

The visited set prevents cycles such as `A -> B -> A` from causing an infinite traversal.

# Problem 12 — Longest Substring Without Repeated Characters

This classic sliding-window problem uses a set to represent the characters currently inside the window.

## Step 1: State the Invariant

At every moment, the active window contains no duplicate characters.

When the next character already exists in the window, move the left boundary forward until the duplicate is removed.

In [47]:
text = "pwwkew"

left = 0
window = set()
best = ""

for right, character in enumerate(text):
    while character in window:
        window.remove(text[left])
        left += 1

    window.add(character)

    current = text[left:right + 1]
    if len(current) > len(best):
        best = current

    print(
        f"right={right}, char={character!r}, "
        f"window={current!r}, best={best!r}"
    )

right=0, char='p', window='p', best='p'
right=1, char='w', window='pw', best='pw'
right=2, char='w', window='w', best='pw'
right=3, char='k', window='wk', best='pw'
right=4, char='e', window='wke', best='wke'
right=5, char='w', window='kew', best='wke'


## Step 2: Turn the Walkthrough Into a Function

In [48]:
def longest_unique_substring(text):
    left = 0
    window = set()
    best_start = 0
    best_length = 0

    for right, character in enumerate(text):
        while character in window:
            window.remove(text[left])
            left += 1

        window.add(character)

        current_length = right - left + 1

        if current_length > best_length:
            best_start = left
            best_length = current_length

    return text[best_start:best_start + best_length]


assert longest_unique_substring("abcabcbb") == "abc"
assert longest_unique_substring("bbbbb") == "b"
assert longest_unique_substring("pwwkew") == "wke"
assert longest_unique_substring("") == ""

longest_unique_substring("advanced sets")

'vanced s'

Each character enters and leaves the window at most once.

The expected time complexity is therefore `O(n)`.

# Problem 13 — Create Hashable Domain Objects

Objects stored in a set must be hashable.

For custom classes, equality and hashing must agree:

> If `a == b`, then `hash(a) == hash(b)` must also be true.

## Step 1: Define Identity

Suppose two account objects represent the same account whenever their numeric IDs match.

The display name is descriptive data, not identity.

In [49]:
@dataclass(frozen=True, eq=False)
class Account:
    account_id: int
    display_name: str

    def __eq__(self, other):
        if not isinstance(other, Account):
            return NotImplemented

        return self.account_id == other.account_id

    def __hash__(self):
        return hash(self.account_id)

## Step 2: Store Accounts in a Set

The second account with ID `1` is considered equal to the first one.

In [50]:
accounts = {
    Account(1, "Ada"),
    Account(1, "Ada Lovelace"),
    Account(2, "Grace"),
}

assert len(accounts) == 2
assert Account(1, "Different label") in accounts

accounts

{Account(account_id=1, display_name='Ada'),
 Account(account_id=2, display_name='Grace')}

The class is frozen so the identity field cannot be modified after insertion.

Changing a field involved in hashing while an object is inside a set can corrupt lookup behavior.

# Problem 14 — Benchmark Membership Testing

Sets are implemented using hash tables.

Average-case membership testing is usually much faster than scanning a large list, especially for:

- values near the end,
- and missing values.

## Step 1: Prepare Equivalent Containers

In [51]:
size = 30_000

values_list = list(range(size))
values_set = set(values_list)

assert len(values_list) == len(values_set)

## Step 2: Benchmark Several Cases

We use repeated measurements and report the median.

The exact numbers will vary by machine and Python version.

In [52]:
def median_membership_time(container, target, *, number=1_000, repeats=5):
    timings = repeat(
        stmt="target in container",
        globals={
            "container": container,
            "target": target,
        },
        number=number,
        repeat=repeats,
    )

    return median(timings)


targets = {
    "near_start": 3,
    "near_end": size - 1,
    "missing": -1,
}

benchmark = {}

for label, target in targets.items():
    list_time = median_membership_time(values_list, target)
    set_time = median_membership_time(values_set, target)

    benchmark[label] = {
        "list_seconds": list_time,
        "set_seconds": set_time,
        "ratio": list_time / set_time,
    }

benchmark

{'near_start': {'list_seconds': 0.00010529998689889908,
  'set_seconds': 5.340017378330231e-05,
  'ratio': 1.971903449719209},
 'near_end': {'list_seconds': 0.3172367997467518,
  'set_seconds': 2.8899870812892914e-05,
  'ratio': 10977.100963552577},
 'missing': {'list_seconds': 0.2647483004257083,
  'set_seconds': 2.240017056465149e-05,
  'ratio': 11819.030558789289}}

## Step 3: Interpret the Result

List membership is a linear scan in the worst case.

Set membership is expected constant time on average.

However, sets generally require more memory, and a list may be competitive when the desired value is almost always near the beginning.

# Problem 15 — Compare Shallow Memory Usage

Performance is not the only consideration.

Sets and dictionaries usually use more container memory than lists because hash tables maintain extra capacity.

## Step 1: Measure the Container Objects

`sys.getsizeof` reports the shallow size of the container object.

It does not recursively add the memory used by every referenced element.

In [53]:
def shallow_sizes(count):
    values = list(range(count))

    return {
        "count": count,
        "list": sys.getsizeof(values),
        "set": sys.getsizeof(set(values)),
        "dict": sys.getsizeof(dict.fromkeys(values)),
    }


for count in (0, 1, 10, 100, 1_000):
    print(shallow_sizes(count))

{'count': 0, 'list': 56, 'set': 216, 'dict': 64}
{'count': 1, 'list': 72, 'set': 216, 'dict': 224}
{'count': 10, 'list': 136, 'set': 728, 'dict': 352}
{'count': 100, 'list': 856, 'set': 8408, 'dict': 4688}
{'count': 1000, 'list': 8056, 'set': 32984, 'dict': 36952}


## Step 2: Notice the Jumps

Container sizes do not necessarily grow after every insertion.

Python over-allocates space so it does not need to resize on every operation.

The exact resizing strategy is implementation-specific.

# Problem 16 — Capstone: Detect Suspicious Account Sharing

A service records the device IDs used by each account.

We want to identify pairs of accounts that share at least a minimum number of devices.

This problem combines:

- sets,
- intersections,
- combinations,
- and Jaccard similarity.

## Step 1: Create Account-to-Device Sets

In [54]:
account_devices = {
    "account-A": {"phone-1", "laptop-1", "tablet-1"},
    "account-B": {"phone-1", "laptop-1", "console-1"},
    "account-C": {"phone-9", "laptop-9"},
    "account-D": {"phone-1", "laptop-1", "tablet-1", "console-1"},
}

## Step 2: Compare One Pair

The intersection gives shared devices.

The union is needed for the similarity score.

In [55]:
devices_a = account_devices["account-A"]
devices_b = account_devices["account-B"]

shared = devices_a & devices_b
similarity = jaccard_similarity(devices_a, devices_b)

print("Shared:", sorted(shared))
print("Similarity:", similarity)

Shared: ['laptop-1', 'phone-1']
Similarity: 0.5


## Step 3: Compare Every Pair of Accounts

`itertools.combinations` produces each pair exactly once.

In [56]:
def suspicious_account_pairs(
    account_devices,
    *,
    minimum_shared=2,
    minimum_similarity=0.5,
):
    normalized = {
        account_id: set(devices)
        for account_id, devices in account_devices.items()
    }

    findings = []

    for left_id, right_id in combinations(normalized, 2):
        left_devices = normalized[left_id]
        right_devices = normalized[right_id]

        shared = left_devices & right_devices
        similarity = jaccard_similarity(
            left_devices,
            right_devices,
        )

        if (
            len(shared) >= minimum_shared
            and similarity >= minimum_similarity
        ):
            findings.append({
                "accounts": (left_id, right_id),
                "shared_devices": shared,
                "shared_count": len(shared),
                "jaccard_similarity": similarity,
            })

    return findings


findings = suspicious_account_pairs(account_devices)

findings

[{'accounts': ('account-A', 'account-B'),
  'shared_devices': {'laptop-1', 'phone-1'},
  'shared_count': 2,
  'jaccard_similarity': 0.5},
 {'accounts': ('account-A', 'account-D'),
  'shared_devices': {'laptop-1', 'phone-1', 'tablet-1'},
  'shared_count': 3,
  'jaccard_similarity': 0.75},
 {'accounts': ('account-B', 'account-D'),
  'shared_devices': {'console-1', 'laptop-1', 'phone-1'},
  'shared_count': 3,
  'jaccard_similarity': 0.75}]

## Step 4: Verify the Findings

Account C should not appear because it shares no devices with the other accounts.

In [57]:
reported_accounts = {
    account_id
    for finding in findings
    for account_id in finding["accounts"]
}

assert "account-C" not in reported_accounts

assert any(
    finding["accounts"] == ("account-A", "account-D")
    and finding["shared_devices"] == {
        "phone-1", "laptop-1", "tablet-1"
    }
    for finding in findings
)

## Step 5: Discuss the Limitations

This example identifies candidates, not proof of account sharing.

Shared devices may have legitimate explanations, such as:

- a family computer,
- a public terminal,
- a corporate network,
- or inaccurate device identification.

Set-based similarity is a useful screening tool, but domain context is still required.

# Further Practice Problems

Try solving these without looking back at the completed solutions.

## Exercise 1

Given sets of users active on seven consecutive days, find:

- users active every day,
- users active on exactly one day,
- users active on at least five days.

## Exercise 2

Build a case-insensitive username registry.

The registry should preserve the first spelling entered, but reject later spellings that normalize to the same username.

## Exercise 3

Given course prerequisites represented as sets, return every course a student is currently eligible to take.

## Exercise 4

Implement a set-backed FIFO queue that prevents duplicate jobs while preserving arrival order.

## Exercise 5

Given two nested dictionaries containing lists and sets, recursively convert them into hashable structures so they can be compared or stored in a set.

## Exercise 6

Create a graph function that returns every node whose neighbor set is a superset of a required set.

## Exercise 7

Write property-based checks for these identities:

- `A | B == B | A`
- `A & B == B & A`
- `A ^ B == (A - B) | (B - A)`
- `A == (A & B) | (A - B)`

## Exercise 8

Benchmark stable deduplication using:

- a set plus a list,
- `dict.fromkeys`,
- and a manual list-only scan.

# Final Summary

Sets are especially useful when a problem is about:

- uniqueness,
- membership,
- overlap,
- exclusion,
- reconciliation,
- or unordered identity.

The most readable solutions usually express the domain rule directly with set algebra.

Examples:

- “required but missing” becomes `required - available`,
- “shared by both” becomes `left & right`,
- “present in exactly one” becomes `left ^ right`,
- “contains every required item” becomes `required <= available`,
- “shares nothing” becomes `left.isdisjoint(right)`.

Sets are powerful, but they are not replacements for every collection.

Use lists, deques, heaps, and dictionaries when order, duplicates, priority, or associated values are central to the problem.